# Group 4 — Data Modelling

Cleaning, target definition, feature selection, train/test split, and imbalance handling, built as a reusable `sklearn` Pipeline. Logic lives in `../KSI.py` — this notebook runs it and documents the reasoning.

**Note:** the categorical column lists below include placeholders for Group 1 (location/conditions) and Group 2 (people/vehicles) columns. Once Aboud and Ibrahim report missing-data % and cardinality for their columns, come back and drop/adjust anything that's mostly empty or has too many unique values to one-hot encode sensibly.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('../data/TOTAL_KSI_4115794401574937330.csv', encoding='utf-8-sig')
df.shape

(18957, 54)

## Drop low-value / identifier / redundant columns

`INDEX`, `ACCNUM`, `OBJECTID` are just record/event identifiers with no predictive value. `OFFSET` is a location-precision detail, not useful on its own. `HOOD_140` / `NEIGHBOURHOOD_140` are the old versions of `HOOD_158` / `NEIGHBOURHOOD_158` — keeping both would duplicate the same information. `x` / `y` are just projected copies of `LATITUDE` / `LONGITUDE`.

In [2]:
drop_cols = [
    'INDEX', 'ACCNUM', 'OBJECTID', 'OFFSET',
    'HOOD_140', 'NEIGHBOURHOOD_140',
    'x', 'y',
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

## Define target

`ACCLASS` has 4 real values: `Fatal`, `Non-Fatal Injury`, `Property Damage O` (truncated "Property Damage Only"), and blank.

- Blank rows are dropped — no usable label.
- Binary target: **1 = Fatal, 0 = everything else** (Non-Fatal Injury and Property Damage Only both count as "not fatal").
- Assumption to confirm with the group: is lumping Property Damage Only in with Non-Fatal Injury the right call, or should those rows be excluded entirely since this is supposed to be a KSI (Killed/Seriously Injured) dataset? They exist because the data is per-*person*, not per-collision — a KSI collision can still have an uninjured party in it.

In [3]:
target_col = 'ACCLASS'
df = df[df[target_col].notna() & (df[target_col].str.strip() != '')]
df[target_col] = df[target_col].apply(lambda x: 1 if str(x).strip().lower() == 'fatal' else 0)

y = df[target_col]
X = df.drop(columns=[target_col])
print('Fatal rate:', y.mean())

Fatal rate: 0.14085250052753745


## Column groups for encoding

Group 3 (Aidan) is verified against the real data — clean `Yes`/blank values, low cardinality. Group 1 and Group 2 lists are placeholders pending their exploration findings.

In [4]:
categorical_cols = [
    # Group 1 (location/conditions) — PLACEHOLDER, pending Aboud's findings
    'ROAD_CLASS', 'DISTRICT', 'ACCLOC', 'TRAFFCTL', 'VISIBILITY',
    'LIGHT', 'RDSFCOND', 'DIVISION', 'NEIGHBOURHOOD_158',
    # Group 2 (people/vehicles) — PLACEHOLDER, pending Ibrahim's findings
    'INVTYPE', 'INVAGE', 'INJURY', 'INITDIR', 'VEHTYPE', 'MANOEUVER',
    'DRIVACT', 'DRIVCOND', 'PEDTYPE', 'PEDACT', 'PEDCOND',
    'CYCLISTYPE', 'CYCACT', 'CYCCOND',
    # Group 3 (Yes/No flags) — reviewed against real data
    'PEDESTRIAN', 'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK',
    'TRSN_CITY_VEH', 'EMERG_VEH', 'PASSENGER', 'SPEEDING', 'AG_DRIV',
    'REDLIGHT', 'ALCOHOL', 'DISABILITY',
]
numeric_cols = ['LATITUDE', 'LONGITUDE']

categorical_cols = [c for c in categorical_cols if c in X.columns]
numeric_cols = [c for c in numeric_cols if c in X.columns]

## Preprocessing pipeline, train/test split, imbalance note

In [5]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, categorical_cols),
    ('num', numeric_transformer, numeric_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

full_pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_test_processed = full_pipeline.transform(X_test)

print('Train shape:', X_train_processed.shape)
print('Test shape:', X_test_processed.shape)
print('Class balance (train):')
print(y_train.value_counts(normalize=True))

Train shape: (15164, 456)
Test shape: (3792, 456)
Class balance (train):
ACCLASS
0    0.85914
1    0.14086
Name: proportion, dtype: float64


## Class imbalance

Fatal collisions are ~14% of the dataset — a real but moderate imbalance, not extreme. Two options for Part 2 (model building):

- **`class_weight='balanced'`** on the classifier itself (simplest, no data duplication) — likely sufficient here given the imbalance isn't severe.
- **SMOTE** (`imbalanced-learn`) if `class_weight` alone doesn't get recall high enough on the fatal class.

Decision deferred to Part 2 once we can compare both against real model metrics.